In [6]:
# codigo inicial que unificou as bases de vendas e vistoria, criou as colunas de ágio absoluto e percentual, 
# e removeu os registros duplicados dos itens agrupados
# removeu os imoveis da tipologia apartamento/nao é caracteristica da empresa. Venda por convenio de outra instituicao
# teste usando o codigo da destinacao original, via onehot encoding

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import Ridge
from sklearn.ensemble import ExtraTreesRegressor, HistGradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR

from sklearn.metrics import (
    mean_absolute_error, r2_score, mean_absolute_percentage_error,
    root_mean_squared_error
)

import matplotlib.pyplot as plt

tabela = pd.read_csv("base_vendas_atividade2_final1.csv", sep=";", encoding="latin-1")
vistoria = pd.read_csv("base_vistorias_atividade2_final.csv", sep=";", encoding="latin-1")
destinacoes = pd.read_csv("base_destinacoes_atividade2_final.csv", sep=";", encoding="latin-1")

tabela = tabela.merge(
    vistoria[["CD_IMOVEL_URBANO", "SIM_SITIMO_DS"]],
    how="left",
    left_on="CD_IMOVEL",
    right_on="CD_IMOVEL_URBANO"
)


tabela = tabela.merge(
    destinacoes[["COD_DESTINACAO_IMOVEL", "TIPO_USO_DESTINACAO", "RESIDENCIAL", "COMERCIAL", "INDUSTRIAL", "INSTITUCIONAL"]],
    how="left",
    left_on="COD_DESTINACAO_IMOVEL",
    right_on="COD_DESTINACAO_IMOVEL"
)       


# # remove a coluna duplicada da chave
tabela = tabela.drop(columns=["CD_IMOVEL_URBANO", "CD_IMOVEL"])
tabela = tabela.rename(columns={"SIM_SITIMO_DS": "SITUACAO_VISTORIA"})
# tabela["SITUACAO_VISTORIA"] = tabela["SITUACAO_VISTORIA"].fillna("SEM_VISTORIA")


# # colunas que definem a duplicidade: mesmo ano, mesmo edital e mesmo item do edital (item de edital com imoveis agrupados)
cols = ["ANO_VENDA", "NR_EDITAL", "ITEM_EDITAL"]
# # máscara: True para linhas que aparecem em duplicidade (em qualquer posição do grupo)
mask_dup = tabela.duplicated(subset=cols, keep=False)
# # remove TODAS as linhas duplicadas desses grupos
tabela_sem_dups = tabela.loc[~mask_dup].copy()

# # Foram removidos 145 itens que constavam como items agrupados, de um total de 2982, restando 2837 registros 
print("Linhas originais:", len(tabela))
print("Linhas removidas:", mask_dup.sum())
print("Linhas finais:", len(tabela_sem_dups))

tabela = tabela_sem_dups
tabela["NR_EDITAL"] = tabela["ANO_VENDA"].astype(str) + "-" + tabela["NR_EDITAL"].astype(str)

def br_to_float(s):
    """
    Converte número no formato BR para float:
    - remove separador de milhar (.)
    - troca decimal (,) por (.)
    """
    if pd.isna(s):
        return np.nan
    s = str(s).strip()
    if s == "":
        return np.nan
    s = s.replace(".", "")      # remove milhares
    s = s.replace(",", ".")     # troca decimal
    return pd.to_numeric(s, errors="coerce")

cols = ["VALOR_VENDA", "AREA_MAX_CONSTR", "AREA_BASE", "AREA", "VALOR_LAUDO"]  # ajuste
for c in cols:
    if c in tabela.columns:
        tabela[c] = tabela[c].apply(br_to_float)

# #criacao da coluna AGIO ABSOLUTO , esta coluna deve ser retirada do treino
tabela["AGIO_ABSOLUTO"] = tabela["VALOR_VENDA"] - tabela["VALOR_LAUDO"]
tabela["AGIO_PERCENTUAL"] = ((tabela["VALOR_VENDA"] - tabela["VALOR_LAUDO"]) / tabela["VALOR_LAUDO"]) * 100

#retirando os apartamentos da base (14 registros) que tem área máxima de construção igual a zero e área base igual a zero, ou seja, não tem área construída, o que é um erro de cadastro
tabela = tabela[~((tabela["AREA_MAX_CONSTR"] == 0) & (tabela["AREA_BASE"] == 0))]   

tabela.info()






Linhas originais: 2982
Linhas removidas: 145
Linhas finais: 2837
<class 'pandas.DataFrame'>
Index: 2823 entries, 0 to 2981
Data columns (total 21 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   DS_CIDADE              2823 non-null   str    
 1   DS_SETOR               2823 non-null   str    
 2   COD_DESTINACAO_IMOVEL  2823 non-null   int64  
 3   ANO_VENDA              2823 non-null   int64  
 4   NR_EDITAL              2823 non-null   str    
 5   ITEM_EDITAL            2823 non-null   int64  
 6   VALOR_LAUDO            2823 non-null   int64  
 7   PERCENTUAL_ENTRADA     2823 non-null   int64  
 8   VALOR_VENDA            2823 non-null   float64
 9   AREA_MAX_CONSTR        2823 non-null   float64
 10  AREA_BASE              2823 non-null   float64
 11  AREA                   2823 non-null   float64
 12  QTD_OFERTAS            2823 non-null   int64  
 13  SITUACAO_VISTORIA      2823 non-null   str    
 14  TIPO_US

In [ ]:
# -------------------------
# 0) DADOS (ajuste se necessário)
# -------------------------
df = tabela.copy()
df.columns = df.columns.str.strip()

TARGET = "VALOR_VENDA"


# Remover colunas que você NÃO quer usar
# (vazamento/colinearidade/dado futuro)
DROP_COLS = ["AGIO_ABSOLUTO", "AGIO_PERCENTUAL", "VALOR_LAUDO", "QTD_OFERTAS", "DS_DESTINACAO_IMOVEL", "COD_DESTINACAO_IMOVEL"]
DROP_COLS = [c for c in DROP_COLS if c in df.columns]

# Percentual 0–100 -> 0–1
if "PERCENTUAL_ENTRADA" in df.columns:
    df["PERCENTUAL_ENTRADA"] = pd.to_numeric(df["PERCENTUAL_ENTRADA"], errors="coerce") / 100.0


# Separar X e y
y = df[TARGET].copy()
X = df.drop(columns=[TARGET] + DROP_COLS)

# DS_CIDADE;DS_SETOR;COD_DESTINACAO_IMOVEL;TIPO_USO_DESTINACAO;
# Definir colunas numéricas e categóricas explicitamente
num_cols = [
    "AREA_MAX_CONSTR", "AREA_BASE", "AREA", "PERCENTUAL_ENTRADA", "RESIDENCIAL", "COMERCIAL", "INDUSTRIAL", "INSTITUCIONAL"]

num_cols = [c for c in num_cols if c in X.columns]

cat_cols = ["DS_CIDADE", "DS_SETOR", "SITUACAO_VISTORIA", "ANO_VENDA", "NR_EDITAL", "TIPO_USO_DESTINACAO"]
cat_cols = [c for c in cat_cols if c in X.columns]

# Tipos (evita problemas de mixed types)
for c in cat_cols:
    X[c] = X[c].astype("string")
for c in num_cols:
    X[c] = pd.to_numeric(X[c], errors="coerce")

print("Numéricas:", num_cols)
print("Categóricas:", cat_cols)
print("Shape X:", X.shape, "| Shape y:", y.shape)

Numéricas: ['AREA_MAX_CONSTR', 'AREA', 'PERCENTUAL_ENTRADA', 'RESIDENCIAL', 'COMERCIAL', 'INDUSTRIAL', 'INSTITUCIONAL']
Categóricas: ['DS_CIDADE', 'DS_SETOR', 'SITUACAO_VISTORIA', 'ANO_VENDA', 'NR_EDITAL', 'TIPO_USO_DESTINACAO']
Shape X: (2823, 14) | Shape y: (2823,)


In [ ]:
# ============================================================
# 2) SPLIT + LOG DO ALVO
# ============================================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

y_train_log = np.log1p(y_train)
y_test_log  = np.log1p(y_test)

# ============================================================
# 3) PREPROCESS (sparse e dense)
# ============================================================
numeric_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

cat_pipe_sparse = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=True)),
])

preprocess_sparse = ColumnTransformer(
    transformers=[
        ("num", numeric_pipe, num_cols),
        ("cat", cat_pipe_sparse, cat_cols),
    ],
    remainder="drop"
)

# Dense (KNN, SVR e aqui também para HistGB)
cat_pipe_dense = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

preprocess_dense = ColumnTransformer(
    transformers=[
        ("num", numeric_pipe, num_cols),
        ("cat", cat_pipe_dense, cat_cols),
    ],
    remainder="drop"
)

# ============================================================
# 4) HIPERPARÂMETROS FIXOS (os que você reportou)
# ============================================================
# Ridge best params: alpha=300, fit_intercept=True
pipe_ridge = Pipeline([
    ("preprocess", preprocess_sparse),
    ("model", Ridge(alpha=300, fit_intercept=True, random_state=42))
])

# ExtraTrees Stage2 best params:
# max_depth=25, max_features=0.7, min_samples_leaf=1, min_samples_split=2, n_estimators=1200
pipe_et = Pipeline([
    ("preprocess", preprocess_sparse),
    ("model", ExtraTreesRegressor(
        random_state=42,
        n_jobs=-1,
        n_estimators=1200,
        max_depth=25,
        max_features=0.7,
        min_samples_leaf=1,
        min_samples_split=2,
    ))
])

# HistGradientBoosting best params:
# l2_regularization=0.1, learning_rate=0.06, max_depth=None, max_iter=300, min_samples_leaf=10
pipe_hgb = Pipeline([
    ("preprocess", preprocess_dense),
    ("model", HistGradientBoostingRegressor(
        random_state=42,
        l2_regularization=0.1,
        learning_rate=0.06,
        max_depth=None,
        max_iter=300,
        min_samples_leaf=10
    ))
])

# DecisionTreeRegressor best params:
# max_depth=None, min_samples_leaf=2, min_samples_split=10
pipe_dt = Pipeline([
    ("preprocess", preprocess_sparse),
    ("model", DecisionTreeRegressor(
        random_state=42,
        max_depth=None,
        min_samples_leaf=2,
        min_samples_split=10
    ))
])

# KNeighborsRegressor best params:
# n_neighbors=5, p=1, weights='distance'
pipe_knn = Pipeline([
    ("preprocess", preprocess_dense),
    ("model", KNeighborsRegressor(
        n_neighbors=5,
        p=1,
        weights="distance"
    ))
])

# SVR best params:
# C=10, epsilon=0.05, gamma='scale'
pipe_svr = Pipeline([
    ("preprocess", preprocess_dense),
    ("model", SVR(
        kernel="rbf",
        C=10,
        epsilon=0.05,
        gamma="scale"
    ))
])

models = {
    "DecisionTreeRegressor": pipe_dt,
    "KNeighborsRegressor": pipe_knn,
    "SVR (RBF)": pipe_svr,
    "Ridge": pipe_ridge,
    "ExtraTrees (Stage2)": pipe_et,
    "HistGradientBoosting": pipe_hgb
}

# ============================================================
# 5) FUNÇÃO PARA MÉTRICAS TREINO/TESTE
# ============================================================
def eval_train_test(pipe, name, piso_mape=100_000):
    pipe.fit(X_train, y_train_log)

    # TREINO
    pred_log_tr = pipe.predict(X_train)
    pred_tr = np.expm1(pred_log_tr)

    mask_tr = y_train >= piso_mape
    mape_tr_piso = mean_absolute_percentage_error(y_train[mask_tr], pred_tr[mask_tr]) if mask_tr.any() else np.nan

    # TESTE
    pred_log_te = pipe.predict(X_test)
    pred_te = np.expm1(pred_log_te)

    mask_te = y_test >= piso_mape
    mape_te_piso = mean_absolute_percentage_error(y_test[mask_te], pred_te[mask_te]) if mask_te.any() else np.nan

    out = {
        "Modelo": name,

        "MAE_R$_treino": mean_absolute_error(y_train, pred_tr),
        "RMSE_R$_treino": root_mean_squared_error(y_train, pred_tr),
        "R2_R$_treino": r2_score(y_train, pred_tr),
        "MAPE_R$_treino": mean_absolute_percentage_error(y_train, pred_tr),
        f"MAPE_R$_>={piso_mape}_treino": mape_tr_piso,
        "RMSE_log_treino": root_mean_squared_error(y_train_log, pred_log_tr),

        "MAE_R$_teste": mean_absolute_error(y_test, pred_te),
        "RMSE_R$_teste": root_mean_squared_error(y_test, pred_te),
        "R2_R$_teste": r2_score(y_test, pred_te),
        "MAPE_R$_teste": mean_absolute_percentage_error(y_test, pred_te),
        f"MAPE_R$_>={piso_mape}_teste": mape_te_piso,
        "RMSE_log_teste": root_mean_squared_error(y_test_log, pred_log_te),
    }
    return out

# ============================================================
# 6) RODAR E MOSTRAR RESUMO
# ============================================================
results = []
for name, pipe in models.items():
    results.append(eval_train_test(pipe, name))

res_df = pd.DataFrame(results).sort_values("MAE_R$_teste", ascending=True)

print("\n=== RESUMO (ordenado por MAE no teste) ===")
print(res_df)


=== RESUMO (ordenado por MAE no teste) ===
                  Modelo  MAE_R$_treino  RMSE_R$_treino  R2_R$_treino  \
4    ExtraTrees (Stage2)   1.909487e+04    1.047289e+05  9.998681e-01   
5   HistGradientBoosting   2.133707e+05    4.776145e+06  7.256411e-01   
0  DecisionTreeRegressor   3.226388e+05    6.161370e+06  5.434182e-01   
2              SVR (RBF)   8.882397e+04    5.059797e+05  9.969209e-01   
1    KNeighborsRegressor   1.145620e+04    1.046729e+05  9.998682e-01   
3                  Ridge   2.339075e+08    1.108094e+10 -1.476785e+06   

   MAPE_R$_treino  MAPE_R$_>=100000_treino  RMSE_log_treino  MAE_R$_teste  \
4        0.041436                 0.037032         0.073049  2.345083e+05   
5        0.115919                 0.108157         0.161670  3.223199e+05   
0        0.121262                 0.117684         0.187159  3.434522e+05   
2        0.104193                 0.091925         0.182653  3.588483e+05   
1        0.022926                 0.019461         0.062574